# Email / SMS Spam Detection with Machine Learning
OIBSIP · Data Science · Task 4

**Goal:** classify messages as *spam* or *ham* (legitimate).

**Dataset:** SMS Spam Collection (Kaggle: search *SMS spam collection dataset*, or UCI). Save it as `spam.csv` next to this notebook. The Kaggle version has columns `v1` (label) and `v2` (text).

In [ ]:
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 1. Load data and check class balance

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
df = df.dropna().drop_duplicates()
print("Shape:", df.shape)
counts = df["label"].value_counts()
print(counts)
print((counts / counts.sum() * 100).round(1).astype(str) + " %")
sns.countplot(data=df, x="label")
plt.title("Spam vs ham")
plt.show()

The classes are imbalanced (far more ham than spam), so accuracy alone is misleading. We will also look at precision, recall and F1.

## 2. Text preprocessing
Lowercase, remove punctuation and digits, remove stop words.

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

df["clean"] = df["text"].apply(preprocess)
df["target"] = (df["label"] == "spam").astype(int)
df[["text", "clean"]].head()

## 3. TF-IDF features
**TF-IDF** (term frequency x inverse document frequency) scores how important a word is to one message relative to all messages. Words that appear often in a message but rarely across the dataset (like *winner* or *prize*) get high weights; common words get low weights.

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df["clean"], df["target"], test_size=0.2, random_state=42, stratify=df["target"])

tfidf = TfidfVectorizer(max_features=5000)
X_train = tfidf.fit_transform(X_train_txt)
X_test = tfidf.transform(X_test_txt)
print(X_train.shape, X_test.shape)

## 4. Train two classifiers

In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}
rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred)})
    print("=" * 55); print(name); print(classification_report(y_test, pred, target_names=["ham", "spam"]))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index("Model").round(4)

## 5. Why recall matters for spam detection
Recall is the share of real spam that we catch. A low-recall filter lets spam (and phishing) reach the inbox. However, precision matters too, because wrongly flagging a legitimate message (false positive) can hide something important. In practice we want high recall without sacrificing precision, which is why we report F1 as well.

## 6. Try your own message

In [ ]:
def predict_message(msg, model=models["Multinomial Naive Bayes"]):
    vec = tfidf.transform([preprocess(msg)])
    return "SPAM" if model.predict(vec)[0] == 1 else "HAM"

print(predict_message("Congratulations! You won a free prize, call now to claim"))
print(predict_message("Are we still meeting for lunch tomorrow?"))

## 7. Conclusion
Summarise which model had the better recall/F1 and which you would deploy. (Optional bonus: add WordCloud plots for spam and ham words using the `wordcloud` package.)